# Arnowitt-Deser-Misner (ADM) 3+1 Background

## Author: Zach Etienne

To run the whole notebook, click the `>>` toolbar button and choose
**Restart Kernel and Run All Cells...**.

This notebook inspects analytic ADM initial data for three simple
numerical-relativity examples. It identifies the spatial metric, extrinsic
curvature, lapse, shift, and shift-driver data, then validates the advertised
component structure with symbolic residuals.

**Notebook Status:** Validated from a fresh kernel.

**Validation Notes:** The final validation cells check Kasner diagonal
structure, Brill-Lindquist conformal flatness, and static-trumpet lapse and
shift formulas with zero symbolic residuals.

### Required Reading

- [ADM 3+1 formulation overview](https://arxiv.org/abs/gr-qc/0405109): the
  space-plus-time split used in the line element below.
- This notebook defines the tensor-index conventions it uses, so no previous
  NRPy tensor notebook is required.
- Previous route context:
  [Code-Writing Path Choice Guide](../5-infrastructures/backend_choice_guide.ipynb).

### NRPy Source Code

These links point to upstream NRPy source for inspection. The notebook imports
the installed package and does not assume a local source checkout.

- [InitialData_Cartesian.py](https://github.com/nrpy/nrpy/blob/main/nrpy/equations/general_relativity/InitialData_Cartesian.py):
  analytic ADM initial data in Cartesian coordinates.
- [InitialData_Spherical.py](https://github.com/nrpy/nrpy/blob/main/nrpy/equations/general_relativity/InitialData_Spherical.py):
  analytic ADM initial data in spherical coordinates.

Navigation:
[Index](../index.ipynb) |
Previous: [Code-Writing Path Choice Guide](../5-infrastructures/backend_choice_guide.ipynb) |
Next: [ADM and BSSN Conversions](adm_bssn_conversions.ipynb)

# Table of Contents

1. [Notation and Terms](#Notation-and-Terms)
1. [Step 1: Read the ADM line element](#Step-1:-Read-the-ADM-line-element)
1. [Step 2: Inspect Kasner diagonal expansion](#Step-2:-Inspect-Kasner-diagonal-expansion)
1. [Validation Check: Kasner diagonal ADM structure](#Validation-Check:-Kasner-diagonal-ADM-structure)
1. [Step 3: Inspect Brill-Lindquist conformal flatness](#Step-3:-Inspect-Brill-Lindquist-conformal-flatness)
1. [Validation Check: Brill-Lindquist isotropy and zero curvature](#Validation-Check:-Brill-Lindquist-isotropy-and-zero-curvature)
1. [Step 4: Inspect static-trumpet gauge data](#Step-4:-Inspect-static-trumpet-gauge-data)
1. [Validation Check: Static-trumpet lapse and shift structure](#Validation-Check:-Static-trumpet-lapse-and-shift-structure)
1. [What next?](#What-next?)

# Notation and Terms
### [Back to [top](#Table-of-Contents)]

**Initial-Data Scope Note:** this notebook inspects analytic data on one
spatial slice. It does not solve a time-evolution problem, choose a finite
numerical grid, advance evolved variables with a Method of Lines scheme, or
impose boundary conditions. The lesson contract is symbolic coordinate
domains, analytic ADM fields, and residual diagnostics.

| Name | Meaning in this notebook | Code name |
| --- | --- | --- |
| Dimension | all tensors here have three spatial components | `range(3)` |
| Index range | Latin spatial indices `i,j` run over `0,1,2` | list indices |
| Basis | Cartesian for Kasner and Brill-Lindquist; spherical for static trumpet | coordinate argument |
| Lower indices | covariant tensor components | `gammaDD[i][j]`, `KDD[i][j]` |
| Upper indices | contravariant vector components | `betaU[i]` |
| Symmetry | `gammaDD` and `KDD` are symmetric rank-2 tensors | `gammaDD[i][j] = gammaDD[j][i]` |
| Contraction | repeated upper/lower index pairs are summed | `K = gamma^{ij} K_{ij}` |
| ADM | spacetime split into space plus time | `gammaDD`, `KDD`, `alpha`, `betaU` |
| Spatial metric | distances within one time slice | `gammaDD` |
| Extrinsic curvature | how the slice bends through spacetime | `KDD` |
| Lapse | how fast time advances between neighboring slices | `alpha` |
| Shift | how spatial coordinates slide between slices | `betaU` |
| Shift-driver data | auxiliary shift data passed to later evolution systems | `BU` |
| Residual | expression that should simplify to zero | `sp.simplify(...)` |

# Step 1: Read the ADM line element
### [Back to [top](#Table-of-Contents)]

The ADM split writes spacetime using spatial distances, time advance, and
coordinate sliding:

$$
ds^2 =
-\alpha^2 dt^2
+ \gamma_{ij}\left(dx^i + \beta^i dt\right)
             \left(dx^j + \beta^j dt\right).
$$

For an evolution code, ADM initial data are the input record for one time
slice:

| ADM data | Physical role | What the examples isolate |
| --- | --- | --- |
| `gammaDD` | spatial distances on the slice | diagonal stretching and conformal scale |
| `KDD` | slice bending through spacetime | zero curvature or diagonal bending |
| `alpha` | time advance between slices | simple lapse or trumpet lapse |
| `betaU`, `BU` | coordinate motion and shift-driver data | zero shift or radial shift |

The next cells use three exact data sets to inspect those roles before the
conversion notebook rewrites the same information in BSSN variables.

The next cell imports standard-library output control, SymPy for symbolic
residuals, and the two NRPy initial-data classes used below.

In [1]:
import contextlib
import io

import sympy as sp

from nrpy.equations.general_relativity.InitialData_Cartesian import (
    InitialData_Cartesian,
)
from nrpy.equations.general_relativity.InitialData_Spherical import (
    InitialData_Spherical,
)

# Step 2: Inspect Kasner diagonal expansion
### [Back to [top](#Table-of-Contents)]

Kasner's exact solution ([Kasner 1921](https://ui.adsabs.harvard.edu/abs/1921AmJMa..43..217K/abstract)) is written
in a Cartesian basis and stretches the coordinate axes at different rates. The
slice time is `KASNER_t0`, and the powers `KASNER_p1`, `KASNER_p2`, and
`KASNER_p3` set the anisotropic expansion:

$$
\gamma_{ij} =
\mathrm{diag}(t^{2p_1}, t^{2p_2}, t^{2p_3}),
\qquad
K_{ij} =
\mathrm{diag}(-p_1 t^{2p_1-1}, -p_2 t^{2p_2-1}, -p_3 t^{2p_3-1}).
$$

Inspect the output for three patterns: diagonal metric and curvature entries,
lapse equal to `1`, and zero shift data.

In [2]:
kasner = InitialData_Cartesian("Kasner")
print("Kasner gammaDD diagonal:", [kasner.gammaDD[i][i] for i in range(3)])
print("Kasner KDD diagonal:", [kasner.KDD[i][i] for i in range(3)])
print("Kasner alpha:", kasner.alpha)
print("Kasner betaU:", kasner.betaU)
print("Kasner BU:", kasner.BU)

Kasner gammaDD diagonal: [KASNER_t0**(2*KASNER_p1), KASNER_t0**(2*KASNER_p2), KASNER_t0**(2*KASNER_p3)]
Kasner KDD diagonal: [-KASNER_p1*KASNER_t0**(2*KASNER_p1 - 1), -KASNER_p2*KASNER_t0**(2*KASNER_p2 - 1), -KASNER_p3*KASNER_t0**(2*KASNER_p3 - 1)]
Kasner alpha: 1
Kasner betaU: [0, 0, 0]
Kasner BU: [0, 0, 0]


# Validation Check: Kasner diagonal ADM structure
### [Back to [top](#Table-of-Contents)]

The trusted result is the displayed diagonal Kasner form. The newly inspected
result is NRPy's `InitialData_Cartesian("Kasner")` object. Zero residuals
below mean that every off-diagonal metric and curvature entry vanishes, the
lapse is `1`, and the shift data are zero.

In [3]:
kasner_gamma_offdiag = [
    sp.simplify(kasner.gammaDD[i][j])
    for i in range(3)
    for j in range(3)
    if i != j
]
kasner_K_offdiag = [
    sp.simplify(kasner.KDD[i][j])
    for i in range(3)
    for j in range(3)
    if i != j
]
kasner_alpha_residual = sp.simplify(kasner.alpha - 1)
kasner_shift_residuals = [sp.simplify(component) for component in kasner.betaU]
kasner_BU_residuals = [sp.simplify(component) for component in kasner.BU]

print("Kasner gammaDD off-diagonal residuals:", kasner_gamma_offdiag)
print("Kasner KDD off-diagonal residuals:", kasner_K_offdiag)
print("Kasner alpha - 1 residual:", kasner_alpha_residual)
print("Kasner betaU residuals:", kasner_shift_residuals)
print("Kasner BU residuals:", kasner_BU_residuals)
if (
    kasner_gamma_offdiag != [0] * 6
    or kasner_K_offdiag != [0] * 6
    or kasner_alpha_residual != 0
    or kasner_shift_residuals != [0] * 3
    or kasner_BU_residuals != [0] * 3
):
    raise RuntimeError("Expected Kasner ADM structure residuals to vanish.")

Kasner gammaDD off-diagonal residuals: [0, 0, 0, 0, 0, 0]
Kasner KDD off-diagonal residuals: [0, 0, 0, 0, 0, 0]
Kasner alpha - 1 residual: 0
Kasner betaU residuals: [0, 0, 0]
Kasner BU residuals: [0, 0, 0]


# Step 3: Inspect Brill-Lindquist conformal flatness
### [Back to [top](#Table-of-Contents)]

The [Brill-Lindquist construction](https://ui.adsabs.harvard.edu/abs/1963PhRv..131..471B/abstract)
is a Cartesian, conformally flat black-hole initial-data family:

$$
\gamma_{ij} = \psi^4 \delta_{ij},
\qquad
K_{ij}=0.
$$

This example isolates spatial scale. The representative diagonal metric entry
carries the conformal scale, while curvature and shift data should vanish.

In [4]:
with contextlib.redirect_stdout(io.StringIO()):
    brill_lindquist = InitialData_Cartesian("BrillLindquist")
print("Brill-Lindquist representative gammaDD[0][0]:", brill_lindquist.gammaDD[0][0])
print("Brill-Lindquist representative KDD[0][0]:", brill_lindquist.KDD[0][0])
print("Brill-Lindquist betaU:", brill_lindquist.betaU)
print("Brill-Lindquist BU:", brill_lindquist.BU)

Brill-Lindquist representative gammaDD[0][0]: (BH1_mass/(2*sqrt((-BH1_posn_x + x)**2 + (-BH1_posn_y + y)**2 + (-BH1_posn_z + z)**2)) + BH2_mass/(2*sqrt((-BH2_posn_x + x)**2 + (-BH2_posn_y + y)**2 + (-BH2_posn_z + z)**2)) + 1)**4
Brill-Lindquist representative KDD[0][0]: 0
Brill-Lindquist betaU: [0, 0, 0]
Brill-Lindquist BU: [0, 0, 0]


# Validation Check: Brill-Lindquist isotropy and zero curvature
### [Back to [top](#Table-of-Contents)]

The trusted result is `gamma_ij = psi^4 delta_ij` and `K_ij = 0`. The newly
inspected result is NRPy's Brill-Lindquist data. The metric residuals compare
every full 3-by-3 component with a conformally flat metric using
`gammaDD[0][0]` as the shared scale.

In [5]:
brill_metric_residuals = [
    sp.simplify(
        brill_lindquist.gammaDD[i][j]
        - (brill_lindquist.gammaDD[0][0] if i == j else 0)
    )
    for i in range(3)
    for j in range(3)
]
brill_curvature_residuals = [
    sp.simplify(brill_lindquist.KDD[i][j])
    for i in range(3)
    for j in range(3)
]
brill_shift_residuals = [sp.simplify(component) for component in brill_lindquist.betaU]
brill_BU_residuals = [sp.simplify(component) for component in brill_lindquist.BU]

print("Brill-Lindquist metric conformal-flat residuals:", brill_metric_residuals)
print("Brill-Lindquist KDD residuals:", brill_curvature_residuals)
print("Brill-Lindquist betaU residuals:", brill_shift_residuals)
print("Brill-Lindquist BU residuals:", brill_BU_residuals)
if (
    brill_metric_residuals != [0] * 9
    or brill_curvature_residuals != [0] * 9
    or brill_shift_residuals != [0] * 3
    or brill_BU_residuals != [0] * 3
):
    raise RuntimeError("Expected Brill-Lindquist ADM residuals to vanish.")

Brill-Lindquist metric conformal-flat residuals: [0, 0, 0, 0, 0, 0, 0, 0, 0]
Brill-Lindquist KDD residuals: [0, 0, 0, 0, 0, 0, 0, 0, 0]
Brill-Lindquist betaU residuals: [0, 0, 0]
Brill-Lindquist BU residuals: [0, 0, 0]


# Step 4: Inspect static-trumpet gauge data
### [Back to [top](#Table-of-Contents)]

Dennison and Baumgarte's
[static trumpet data](https://arxiv.org/abs/1403.5484) use a spherical basis:

$$
\alpha=\frac{r}{r+M},
\qquad
\beta^r=\frac{Mr}{(r+M)^2},
\qquad
\beta^\theta=\beta^\phi=0.
$$

This example isolates gauge and coordinate motion. The radial shift is
nonzero, the angular shifts vanish, and `BU` is zero in this analytic data
object.

In [6]:
static_trumpet = InitialData_Spherical("StaticTrumpet")
print("Static trumpet alpha:", static_trumpet.alpha)
print("Static trumpet radial shift betaU[0]:", static_trumpet.betaU[0])
print("Static trumpet angular shifts:", static_trumpet.betaU[1:])
print("Static trumpet BU:", static_trumpet.BU)

Static trumpet alpha: r/(M + r)
Static trumpet radial shift betaU[0]: M*r/(M + r)**2
Static trumpet angular shifts: [0, 0]
Static trumpet BU: [0, 0, 0]


# Validation Check: Static-trumpet lapse and shift structure
### [Back to [top](#Table-of-Contents)]

The trusted result is the displayed lapse and radial-shift formulas. The newly
inspected result is NRPy's `InitialData_Spherical("StaticTrumpet")` object.
Zero residuals mean the analytic lapse, radial shift, angular shifts, and
shift-driver data match the expected structure.

In [7]:
static_symbols = {
    str(symbol): symbol
    for symbol in static_trumpet.alpha.free_symbols | static_trumpet.betaU[0].free_symbols
}
M = static_symbols["M"]
r = static_symbols["r"]
static_alpha_residual = sp.simplify(static_trumpet.alpha - r / (M + r))
static_radial_shift_residual = sp.simplify(
    static_trumpet.betaU[0] - M * r / (M + r) ** 2
)
static_angular_shift_residuals = [
    sp.simplify(static_trumpet.betaU[1]),
    sp.simplify(static_trumpet.betaU[2]),
]
static_BU_residuals = [sp.simplify(component) for component in static_trumpet.BU]

print("Static trumpet alpha residual:", static_alpha_residual)
print("Static trumpet radial shift residual:", static_radial_shift_residual)
print("Static trumpet angular shift residuals:", static_angular_shift_residuals)
print("Static trumpet BU residuals:", static_BU_residuals)
if (
    static_alpha_residual != 0
    or static_radial_shift_residual != 0
    or static_angular_shift_residuals != [0, 0]
    or static_BU_residuals != [0, 0, 0]
):
    raise RuntimeError("Expected static-trumpet lapse and shift residuals to vanish.")

Static trumpet alpha residual: 0
Static trumpet radial shift residual: 0
Static trumpet angular shift residuals: [0, 0]
Static trumpet BU residuals: [0, 0, 0]


# What next?
### [Back to [top](#Table-of-Contents)]

The residual checks verified the ADM structure used by the three examples:
Kasner exercises diagonal expansion and curvature, Brill-Lindquist isolates
conformal spatial scale with zero curvature, and static trumpet shows nonzero
radial coordinate motion.

Reflection prompt: identify which residuals prove metric structure, which
prove zero curvature, and which prove gauge or shift structure.

Next: [ADM and BSSN Conversions](adm_bssn_conversions.ipynb) rewrites these
ADM fields into BSSN variables and checks a symbolic round trip.